# WhisperX for uploading files

Upload your local files to the Colab Files from the left sidebar.

从左侧将视频音频文件上传到Colab，然后运行即可下载生成好的字幕文件。



In [ ]:
# @title *通用参数/Required settings:**
# Check GPU availability
!nvidia-smi

# @markdown #### **Initial prompt**
# @markdown Prompts can be very helpful for correcting specific words or acronyms that the model often misrecognizes in the audio.
prompt = "This is a technical sharing session about data center networking and network operations, primarily in English with some Chinese. Technical terms include NC (Network Cluster), DCN (Data Center Network), MPLS, MPLS-VPN, BGP, eBGP, iBGP, OSPF, IS-IS, MP-BGP, EVPN, VXLAN, VLAN, VRF, ECMP, LACP, Eth-Trunk, Route Reflector, Route Policy, AS-PATH, Next Hop, Prefix, Core, Aggregation, ToR, Leaf, Spine, DCGW, BMC, BMCAGG, AZ, LLD, MOP, Rollback, Migration, Decommission, Failover, Huawei, Ruijie, CloudEngine, CE6865, NE40E, VRP, eSight, MNet, and CloudMNet."  # @param {type:"string"}

# @markdown #### **Directory Path**
# @markdown where your audio-video files are located from Coloab
directory_path = "/content"  # @param {type:"string"}

# @markdown #### Model
model_size = "large-v3"  # @param ["base", "base.en", "small", "small.en","medium", "medium.en", "large-v1","large-v2","large-v3"]

# @markdown #### Language
language = "auto" # @param ["auto", "en", "zh", "de", "es", "ru", "ko", "fr", "ja", "pt", "tr", "pl", "ca", "nl", "ar", "sv", "it", "id", "hi", "fi", "vi", "he", "uk", "el", "ms", "cs", "ro", "da", "hu", "ta", "no", "th", "ur", "hr", "bg", "lt", "la", "mi", "ml", "cy", "sk", "te", "fa", "lv", "bn", "sr", "az", "sl", "kn", "et", "mk", "br", "eu", "is", "hy", "ne", "mn", "bs", "kk", "sq", "sw", "gl", "mr", "pa", "si", "km", "sn", "yo", "so", "af", "oc", "ka", "be", "tg", "sd", "gu", "am", "yi", "lo", "uz", "fo", "ht", "ps", "tk", "nn", "mt", "sa", "lb", "my", "bo", "tl", "mg", "as", "tt", "haw", "ln", "ha", "ba", "jw", "su"]


# @markdown #### Assign speaker labels
# @markdown Recognize speakers
assign_speaker_lable = False # @param {type:"boolean"}

# @markdown #### Align whisper output
align_whisper_output = True # @param {type:"boolean"}

# Install WhisperX

In [ ]:
! pip install git+https://github.com/m-bain/whisperx.git

# ! pip install git+https://github.com/openai/whisper.git


# Get Subtitle Files using WhisperX

### 上传与运行
先运行参数和安装单元，然后把音频或视频上传到 Colab 的 `/content` 根目录，运行处理单元。它会自动扫描并逐个转录支持的文件，实时显示进度和带时间戳的文字。模型加载、音频准备和 VAD 分段阶段可能暂时没有逐段文字。
如果某个文件解码失败，Notebook 会报告错误并继续处理其他文件。本次损坏的原始 `.m4a` 请用从完整 `.mp4` 音轨提取的恢复版代替。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import wave

from google.colab import files, userdata

# 即使只运行本单元，也使用可工作的默认值；先运行参数单元可覆盖它们。
directory_path = globals().get("directory_path", "/content")
model_size = globals().get("model_size", "large-v3")
language = globals().get("language", "auto")
prompt = globals().get("prompt", "")
align_whisper_output = globals().get("align_whisper_output", True)
assign_speaker_lable = globals().get("assign_speaker_lable", False)

EXTENSIONS = {".mp4", ".wav", ".mp3", ".m4a"}
input_dir = Path(directory_path)
prepared_dir = Path("/content/whisperx_prepared")
output_dir = Path("/content/whisperx_results")
prepared_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)


def source_duration(path):
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a:0",
         "-show_entries", "stream=duration",
         "-of", "json", str(path)], capture_output=True, text=True
    )
    if probe.returncode != 0:
        return None
    try:
        return float(json.loads(probe.stdout)["streams"][0]["duration"])
    except (KeyError, IndexError, ValueError, TypeError):
        return None


def prepare_audio(source):
    wav_path = prepared_dir / f"{source.stem}.wav"
    common = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]
    normal = common + ["-i", str(source), "-map", "0:a:0", "-ac", "1",
                       "-ar", "16000", "-c:a", "pcm_s16le", str(wav_path)]
    fallback = common + ["-err_detect", "ignore_err", "-i", str(source),
                         "-map", "0:a:0", "-af", "pan=mono|c0=c0",
                         "-ar", "16000", "-c:a", "pcm_s16le", str(wav_path)]

    for attempt, command in enumerate((normal, fallback), start=1):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0 and wav_path.exists() and wav_path.stat().st_size > 44:
            if attempt == 2:
                print("已使用容错解码，仅保留输入音频的第一个声道；请核听受损位置。")
            break
        wav_path.unlink(missing_ok=True)
        if attempt == 1:
            print("普通解码失败，尝试绕过异常声道布局。FFmpeg 末尾信息：")
            print(result.stderr[-1500:])
        else:
            raise RuntimeError(f"{source.name} 无法解码。FFmpeg 末尾信息：\n{result.stderr[-3000:]}")

    with wave.open(str(wav_path), "rb") as wav:
        if wav.getnchannels() != 1 or wav.getframerate() != 16000:
            raise RuntimeError("生成的 WAV 不是 16 kHz 单声道。")
        decoded_duration = wav.getnframes() / wav.getframerate()
    expected_duration = source_duration(source)
    if expected_duration is not None and expected_duration - decoded_duration > 5:
        wav_path.unlink(missing_ok=True)
        raise RuntimeError(
            f"解码结果只有 {decoded_duration:.1f} 秒，源文件标称 {expected_duration:.1f} 秒；"
            "可能缺失尾段，请检查原始文件。"
        )
    print(f"音频准备完成：{decoded_duration:.1f} 秒，{wav_path}")
    return wav_path


def process_file(source):
    wav_path = prepare_audio(source)
    command = [sys.executable, "-u", "-m", "whisperx", str(wav_path),
               "--model", model_size, "--output_dir", str(output_dir),
               "--verbose", "True", "--print_progress", "True"]
    if language != "auto":
        command += ["--language", language]
    if prompt:
        command += ["--initial_prompt", prompt]
    if not align_whisper_output:
        command.append("--no_align")
    # 对齐时采用 WhisperX 按识别语言自动选择的模型，不强制指定英文模型。
    if assign_speaker_lable:
        token = userdata.get("HF_TOKEN")
        if not token:
            raise RuntimeError("请在 Colab Secrets 中设置 HF_TOKEN 后再启用说话人识别。")
        command += ["--diarize", "--hf_token", token]

    # 逐行转发输出，让 Colab 边运行边显示百分比和带时间戳的转录文本。
    with subprocess.Popen(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    for ext in (".srt", ".json"):
        result_file = output_dir / f"{source.stem}{ext}"
        if not result_file.is_file():
            raise FileNotFoundError(f"WhisperX 已结束，但未生成 {result_file}")
        files.download(str(result_file))


# 自动处理 /content 根目录中的音频和视频；生成文件位于子目录，不会重复扫描。
inputs = sorted(p for p in input_dir.iterdir()
                if p.is_file() and p.suffix.lower() in EXTENSIONS)
if not inputs:
    raise FileNotFoundError(f"{input_dir} 中没有可处理的音频或视频文件。")
failures = []
successes = 0
for source in inputs:
    print(f"开始处理：{source.name}", flush=True)
    try:
        process_file(source)
        successes += 1
    except Exception as exc:
        failures.append((source.name, str(exc)))
        print(f"处理失败：{source.name}: {exc}", flush=True)
if failures:
    print(f"完成 {successes} 个文件，失败 {len(failures)} 个文件。")
if successes == 0:
    raise RuntimeError("没有文件成功转录；请检查上面的错误。")
